# Exp 10: EfficientNet

Card: [exp10_efficientnet.md](../../docs/experiments/exp10_efficientnet.md)


In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from core.training.config import ROOT
from core.training.config import VIP
from core.training.cv import CVResult
from core.training.runs import read_config
from core.analysis.gradcam import GradCamFigures
from core.training.tracking import start_run
from core.analysis.plots import pr_curves, save_figures, use_style

N_SHOW = 4
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp10_efficientnet" / "efficientnet_b0.yaml"
BASELINE = CONFIG_DIR / "exp09_full_ratio_weight" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp10_efficientnet"

# dataviz reference palette, light mode; categorical slots in fixed order
SERIES = {"ResNet-18 (exp09)": "#2a78d6", "EfficientNet-B0": "#1baf7a"}
use_style()

## Training


In [ ]:
results = {"ResNet-18 (exp09)": CVResult.load(BASELINE)}

In [ ]:
results["EfficientNet-B0"] = CVResult.load(CONFIG)

## Training curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for key, ax in zip(["train_map", "val_map"], axes):
    for name, r in results.items():
        ax.plot(
            np.arange(1, len(r.mean_curve("val_map")) + 1),
            r.mean_curve(key),
            color=SERIES[name],
            lw=2,
            label=name,
        )
    ax.set(title=f"{key}, mean over folds", xlabel="epoch")
axes[1].legend(loc="lower right", title="model", alignment="left")
fig.tight_layout()
figures = {f"{PREFIX}_training.png": fig}
plt.show()

## AP and Macro F1


In [ ]:
scores = pd.concat(
    {name: r.fold_scores() for name, r in results.items()}, names=["model"]
)
scores.groupby("model", sort=False).mean().round(3)

In [ ]:
# counts only if same sign on both folds
diff = (
    results["EfficientNet-B0"].fold_scores() - results["ResNet-18 (exp09)"].fold_scores()
).T.add_prefix("fold ")
diff["same sign"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Precision-recall curves


In [ ]:
fig = pr_curves(results, VIP, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with start_run(read_config(CONFIG)["experiment"], "summary", {"val_mode": "fast"}):
    mlflow.log_table(scores.reset_index(), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")

save_figures(figures, FIGURES)